In [1]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
files = glob.glob("../data/cleaned/*.csv")

len(files), files

(8,
 ['../data/cleaned\\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.csv',
  '../data/cleaned\\Friday-WorkingHours-Afternoon_cleaned.csv',
  '../data/cleaned\\Friday-WorkingHours-Morning.pcap_ISCX_cleaned.csv',
  '../data/cleaned\\Monday-WorkingHours.pcap_ISCX_cleaned.csv',
  '../data/cleaned\\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_cleaned.csv',
  '../data/cleaned\\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX_cleaned.csv',
  '../data/cleaned\\Tuesday-WorkingHours.pcap_ISCX_cleaned.csv',
  '../data/cleaned\\Wednesday-workingHours.pcap_ISCX_cleaned.csv'])

In [3]:
dfs = []

for file in files:
    print("Loading:", os.path.basename(file))
    
    df = pd.read_csv(file)
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

print("Combined shape:", combined_df.shape)

Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.csv
Loading: Friday-WorkingHours-Afternoon_cleaned.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX_cleaned.csv
Loading: Monday-WorkingHours.pcap_ISCX_cleaned.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_cleaned.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX_cleaned.csv
Loading: Tuesday-WorkingHours.pcap_ISCX_cleaned.csv
Loading: Wednesday-workingHours.pcap_ISCX_cleaned.csv
Combined shape: (2574264, 79)


In [4]:
combined_df["Label"].value_counts()

Label
BENIGN                        2148386
DoS Hulk                       172849
DDoS                           128016
PortScan                        90819
DoS GoldenEye                   10286
FTP-Patator                      5933
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1953
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

We don't want BENIGN in Model 2 because Model 1 already handles that. and in these we are doing attack classification

In [5]:
attack_df = combined_df[combined_df["Label"] != "BENIGN"].copy()

print("Attack-only shape:", attack_df.shape)

Attack-only shape: (425878, 79)


In [6]:
attack_df["Label"].value_counts()

Label
DoS Hulk                      172849
DDoS                          128016
PortScan                       90819
DoS GoldenEye                  10286
FTP-Patator                     5933
DoS slowloris                   5385
DoS Slowhttptest                5228
SSH-Patator                     3219
Bot                             1953
Web Attack � Brute Force        1470
Web Attack � XSS                 652
Infiltration                      36
Web Attack � Sql Injection        21
Heartbleed                        11
Name: count, dtype: int64

If we train a classifier directly on these, the model gets almost no examples from which to learn their patterns. We don't want to pretend that an 11-sample class can be classified reliably.

So for this project, we'll group attack classes with fewer than 100 samples into: Other
,That means:-
Infiltration,
Web Attack - Sql Injection,
Heartbleed

In [7]:
label_counts = attack_df["Label"].value_counts()

rare_labels = label_counts[label_counts < 100].index

attack_df["AttackType"] = attack_df["Label"].where(
    ~attack_df["Label"].isin(rare_labels),
    "Other"
)

print(attack_df["AttackType"].value_counts())

AttackType
DoS Hulk                    172849
DDoS                        128016
PortScan                     90819
DoS GoldenEye                10286
FTP-Patator                   5933
DoS slowloris                 5385
DoS Slowhttptest              5228
SSH-Patator                   3219
Bot                           1953
Web Attack � Brute Force      1470
Web Attack � XSS               652
Other                           68
Name: count, dtype: int64


So we haven't deleted those attacks. We're grouping them into one modeling category called Other while keeping the original Label column.

Model training starts


In [8]:
X = attack_df.drop(columns=["Label", "AttackType"])
y = attack_df["AttackType"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (425878, 78)
y shape: (425878,)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (340702, 78)
X_test: (85176, 78)
y_train: (340702,)
y_test: (85176,)


stratify=y tries to maintain the same class proportions in both training and testing sets.

check for invalid negative values before training.

In [10]:
negative_columns = [
    col for col in X_train.columns
    if (X_train[col] < 0).any()
]

print("Columns containing negative values:")
print(negative_columns)

Columns containing negative values:
['Flow IAT Min', 'Fwd IAT Min', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward']


In [11]:
for col in negative_columns:
    print(col, ":", (X_train[col] < 0).sum())

Flow IAT Min : 152
Fwd IAT Min : 16
Init_Win_bytes_forward : 7
Init_Win_bytes_backward : 50689


Replace negative values with NaN

In [12]:
X_train = X_train.copy()
X_test = X_test.copy()

for col in negative_columns:
    X_train.loc[X_train[col] < 0, col] = np.nan
    X_test.loc[X_test[col] < 0, col] = np.nan

Fill using training medians

In [13]:
for col in negative_columns:
    median_value = X_train[col].median()
    
    X_train[col] = X_train[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

In [14]:
print("Negative values in X_train:",
      (X_train < 0).sum().sum())

print("Negative values in X_test:",
      (X_test < 0).sum().sum())

print("Missing values in X_train:",
      X_train.isnull().sum().sum())

print("Missing values in X_test:",
      X_test.isnull().sum().sum())

Negative values in X_train: 0
Negative values in X_test: 0
Missing values in X_train: 0
Missing values in X_test: 0


class_weight="balanced" is included because our attack classes are very uneven in size. It tells Random Forest to give more importance to smaller classes during training.

In [15]:
from sklearn.ensemble import RandomForestClassifier

attack_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

attack_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)

In [16]:
import joblib

joblib.dump(attack_model, "../model_attack_classifier.pkl")

print("Model 2 saved successfully!")

Model 2 saved successfully!


In [17]:
y_pred = attack_model.predict(X_test)

In [18]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

Accuracy: 0.9979923922231615
Accuracy (%): 99.79923922231615


In [19]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

                          precision    recall  f1-score   support

                     Bot       1.00      1.00      1.00       391
                    DDoS       1.00      1.00      1.00     25603
           DoS GoldenEye       1.00      1.00      1.00      2057
                DoS Hulk       1.00      1.00      1.00     34570
        DoS Slowhttptest       1.00      1.00      1.00      1046
           DoS slowloris       1.00      1.00      1.00      1077
             FTP-Patator       1.00      1.00      1.00      1187
                   Other       1.00      0.85      0.92        13
                PortScan       1.00      1.00      1.00     18164
             SSH-Patator       1.00      1.00      1.00       644
Web Attack � Brute Force       0.74      0.81      0.77       294
        Web Attack � XSS       0.47      0.36      0.41       130

                accuracy                           1.00     85176
               macro avg       0.93      0.92      0.92     85176
        

So we should not say "Model 2 identifies every attack type with 99.8% performance." The overall accuracy hides the weaker minority classes.

we should generate the confusion matrix for Model 2. That will show us exactly which attack types are getting confused with each other, especially the Web Attack classes.

In [20]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[  391     0     0     0     0     0     0     0     0     0     0     0]
 [    0 25601     0     2     0     0     0     0     0     0     0     0]
 [    0     0  2052     4     1     0     0     0     0     0     0     0]
 [    0     0     5 34565     0     0     0     0     0     0     0     0]
 [    0     0     1     0  1041     3     0     0     0     0     1     0]
 [    0     0     0     0     1  1075     0     0     0     0     1     0]
 [    0     0     0     0     0     0  1187     0     0     0     0     0]
 [    0     0     1     0     0     0     0    11     0     0     1     0]
 [    0     0     0     7     0     1     0     0 18153     0     3     0]
 [    0     0     0     1     0     0     0     0     0   643     0     0]
 [    0     0     0     2     0     0     0     0     0     0   239    53]
 [    0     0     0     1     1     1     0     0     0     0    80    47]]


“The attack-classification model achieved 99.80% overall accuracy and a macro F1-score of 0.92. Most attack categories were classified with high precision and recall. Lower performance was observed for Web Attack–XSS and Web Attack–Brute Force, which showed mutual misclassification in the confusion matrix.”

1)Actual Brute Force: 294

Correctly identified: 239

Misclassified as XSS: 53

2)Actual XSS: 130

Correctly identified: 47

Misclassified as Brute Force: 80